<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-03-prompt-engineering/lesson-3.5-structured-output-and-tool-use/notebooks/exercises-3_5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3.5: Structured Output & Tool Use

*Module 3 · Prompt Engineering & Context Design*

> Free-text AI responses are useless to your app. This lesson teaches you to get clean JSON, validated data, and automatic function calls — the bridge between AI and software.

---

## About this notebook

This notebook contains the **12 runnable code examples** from the published lesson `lesson-3.5.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. `setup.py`
2. Step 1 — Level 1: JSON Output via Prompt Tricks — `part1_json_prompt.py`
3. Step 2 — XML-Tagged Responses — Multiple Sections in One Response — `part2_xml_tags.py`
4. Step 3 — Pydantic Validation Pipeline — Never Trust Raw AI Output — `part3_pydantic_pipeline.py`
5. Step 4 — Gemini JSON Mode — Guaranteed Valid JSON — `part4_json_mode.py`
6. Step 5 — Function Calling — AI Decides Which Tool to Use — `part5_function_calling.py`
7. Step 6 — Project: AI-Powered Product API — `project_product_api.py`
8. Step 7 — The 4-Tier Reliability Hierarchy — Choose the Right Level — `constrained_decoding.py`
9. Step 8 — Claude & OpenAI Structured Outputs — Native APIs — `openai_structured.py`
10. Step 8 — Claude & OpenAI Structured Outputs — Native APIs — `claude_structured.py`
11. Step 9 — The instructor Library — Production-Grade Extraction — `instructor_demo.py`
12. Step 10 — India-Specific Extraction — GSTIN, PAN, Aadhaar & Hinglish — `indian_extraction.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q anthropic>=0.40.0 google-generativeai openai pydantic


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


**`setup.py`** · _Block 1 of 12_


In [ ]:
import google.generativeai as genai
import os, json, re
from pydantic import BaseModel, Field
from typing import Optional

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))


### Step 1 · Level 1: JSON Output via Prompt Tricks

The simplest approach — ask for JSON in your prompt. Works with any model.

**`part1_json_prompt.py`** · _Block 2 of 12_

LEVEL 1: Ask for JSON in the prompt — Works with any model. Simple but not foolproof.


In [ ]:
# =============================================
# LEVEL 1: Ask for JSON in the prompt
# Works with any model. Simple but not foolproof.
# =============================================

model = genai.GenerativeModel("gemini-2.0-flash", generation_config={"temperature": 0.1})

review = "Just got the Boat Airdopes 141. Sound is good for the price, ₹1,099. But the mic quality in calls is terrible — people can't hear me. 3 stars."

# ── The BAD way: vague prompt ──
bad_prompt = f"Extract product info from this review: {review}"

print("BAD prompt (vague):")
print(model.generate_content(bad_prompt).text)
# Returns a paragraph of prose — useless for code!

# ── The GOOD way: show exact JSON shape ──
good_prompt = f"""Extract product information from this review.

Review: "{review}"

Return ONLY a valid JSON object (no markdown, no explanation) with this exact structure:
{{
    "product_name": "string",
    "brand": "string",
    "price": number or null,
    "currency": "INR" or "USD",
    "rating": integer 1-5,
    "sentiment": "positive" | "negative" | "mixed",
    "pros": ["string"],
    "cons": ["string"]
}}"""

print("\nGOOD prompt (structured):")
result = model.generate_content(good_prompt).text.strip()
print(result)

# Parse it!
try:
    # Clean markdown fences if present
    clean = re.sub(r"```json?\s*", "", result).replace("```", "").strip()
    data = json.loads(clean)
    print(f"\n✅ Parsed! Product: {data['product_name']}, ₹{data['price']}")
except json.JSONDecodeError as e:
    print(f"❌ Failed to parse: {e}")


> **What just happened?** The vague prompt returned a paragraph. The structured prompt returned parseable JSON with exact field names and types. The key tricks: (1) show the exact JSON shape, (2) say "ONLY valid JSON", (3) say "no markdown, no explanation." This works ~90% of the time. For the other 10%, we need validation (Part 3) and JSON mode (Part 4).


### Step 2 · XML-Tagged Responses — Multiple Sections in One Response

Sometimes you need the AI to return several different things at once. XML tags keep them separated.

**`part2_xml_tags.py`** · _Block 3 of 12_

XML TAGS: Get multiple sections in one response — Much easier to parse than free text!


In [ ]:
# =============================================
# XML TAGS: Get multiple sections in one response
# Much easier to parse than free text!
# =============================================

model = genai.GenerativeModel("gemini-2.0-flash", generation_config={"temperature": 0.2})

article = """India's UPI system processed 14.04 billion transactions worth ₹20.64 lakh 
crore in December 2024, marking a 45% year-over-year growth. PhonePe leads 
with 48% market share, followed by Google Pay at 37%. The RBI plans to expand 
UPI connectivity to 20 more countries in 2025."""

prompt = f"""Analyze this article and return your analysis using these XML tags:

<summary>A 2-sentence summary of the article</summary>

<entities>A JSON array of named entities: [{{"name": "...", "type": "ORG|PERSON|MONEY|DATE"}}]</entities>

<sentiment>positive, negative, or neutral</sentiment>

<keywords>Comma-separated list of 5 keywords</keywords>

<follow_up>One question a reader might ask after reading this</follow_up>

Article: {article}"""

response = model.generate_content(prompt).text
print("Raw response:")
print(response)

# ── Parse each XML section ──
def extract_tag(text: str, tag: str) -> str:
    """Extract content between XML tags."""
    match = re.search(f"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
    return match.group(1).strip() if match else ""

summary = extract_tag(response, "summary")
entities_raw = extract_tag(response, "entities")
sentiment = extract_tag(response, "sentiment")
keywords = extract_tag(response, "keywords")
follow_up = extract_tag(response, "follow_up")

print("\nParsed sections:")
print(f"  Summary:   {summary[:100]}...")
print(f"  Sentiment: {sentiment}")
print(f"  Keywords:  {keywords}")
print(f"  Follow-up: {follow_up}")

# Parse the entities JSON inside the XML tag
try:
    entities = json.loads(entities_raw)
    print(f"  Entities:  {len(entities)} found")
    for e in entities:
        print(f"    - {e['name']} ({e['type']})")
except:
    print(f"  Entities (raw): {entities_raw[:80]}")


> **What just happened?** One API call returned 5 different sections, each wrapped in XML tags. We parsed each section independently: summary as text, entities as JSON (inside the XML tag!), sentiment as a single word, keywords as a comma-separated list. XML tags are perfect when you need multiple output types in one response — some text, some JSON, some categories. Much cleaner than trying to split free text with regex.


### Step 3 · Pydantic Validation Pipeline — Never Trust Raw AI Output

The AI WILL sometimes return bad data. Pydantic catches it. Auto-retry fixes it.

**`part3_pydantic_pipeline.py`** · _Block 4 of 12_

PYDANTIC VALIDATION PIPELINE — Try to parse → if fail → tell AI what's wrong


In [ ]:
# =============================================
# PYDANTIC VALIDATION PIPELINE
# Try to parse → if fail → tell AI what's wrong
# → retry → parse again. Up to 3 attempts.
# =============================================

from pydantic import BaseModel, Field, ValidationError
from typing import Optional, Type
import json, re

class ValidatedOutput:
    """Get structured, validated output from any LLM."""
    
    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries
        self.model = genai.GenerativeModel("gemini-2.0-flash",
            generation_config={"temperature": 0.1})
    
    def _clean_json(self, text: str) -> str:
        """Remove markdown fences and extra text around JSON."""
        text = re.sub(r"```json?\s*", "", text)
        text = text.replace("```", "").strip()
        # Find the JSON object
        start = text.find("{")
        end = text.rfind("}") + 1
        if start >= 0 and end > start:
            return text[start:end]
        return text
    
    def generate(self, prompt: str, schema: Type[BaseModel]) -> BaseModel:
        """Generate output matching a Pydantic schema with auto-retry."""
        
        # Build the initial prompt with schema
        full_prompt = f"""{prompt}

Return ONLY valid JSON matching this schema (no markdown, no extra text):
{json.dumps(schema.model_json_schema(), indent=2)}"""
        
        last_error = None
        
        for attempt in range(self.max_retries):
            # If this is a retry, include the error
            if last_error:
                full_prompt += f"\n\nYour previous response had this error:\n{last_error}\nPlease fix it and return valid JSON."
            
            # Get response from LLM
            response = self.model.generate_content(full_prompt).text.strip()
            clean = self._clean_json(response)
            
            # Try to parse and validate
            try:
                result = schema.model_validate_json(clean)
                print(f"  ✅ Validated on attempt {attempt + 1}")
                return result
            
            except (ValidationError, json.JSONDecodeError) as e:
                last_error = str(e)
                print(f"  ⚠️ Attempt {attempt + 1} failed: {str(e)[:80]}")
        
        raise ValueError(f"Failed after {self.max_retries} attempts. Last error: {last_error}")

# ── Define schemas for different tasks ──

class ProductReview(BaseModel):
    product: str = Field(description="Product name")
    brand: str = Field(description="Brand name")
    price: Optional[float] = Field(default=None, description="Price in INR")
    rating: int = Field(ge=1, le=5, description="Rating 1-5")
    sentiment: str = Field(pattern="^(positive|negative|mixed)$")
    pros: list[str] = Field(min_length=1)
    cons: list[str] = Field(min_length=1)

class MeetingAction(BaseModel):
    task: str = Field(description="What needs to be done")
    assignee: str = Field(description="Who is responsible")
    deadline: str = Field(description="Due date in YYYY-MM-DD format", pattern="^\d{4}-\d{2}-\d{2}$")
    priority: str = Field(pattern="^(high|medium|low)$")

class MeetingNotes(BaseModel):
    title: str
    date: str
    attendees: list[str] = Field(min_length=1)
    summary: str = Field(max_length=300)
    action_items: list[MeetingAction] = Field(min_length=1)
    next_meeting: Optional[str] = None

# ── Test it! ──
validator = ValidatedOutput(max_retries=3)

# Test 1: Product review
print("Test 1: Product Review Extraction")
review_result = validator.generate(
    'Extract info from: "The Realme Buds Air 5 Pro are amazing! ₹3,999 for ANC and great bass. Only issue is the case is bulky. 4/5."',
    ProductReview
)
print(f"  Product: {review_result.product}")
print(f"  Rating: {review_result.rating}/5, Sentiment: {review_result.sentiment}")

# Test 2: Meeting notes (complex nested schema)
print(f"\nTest 2: Meeting Notes Extraction")
meeting_result = validator.generate(
    """Extract structured notes from this meeting transcript:
    
"Team standup on Jan 15, 2026. Present: Priya, Arjun, Meera. 
Priya said the homepage redesign is on track for Jan 22 launch. 
Arjun needs to fix the payment bug by Jan 18 — high priority. 
Meera will prepare the investor deck by Jan 20, medium priority.
Next standup: Jan 16, 2026."
""",
    MeetingNotes
)
print(f"  Title: {meeting_result.title}")
print(f"  Attendees: {meeting_result.attendees}")
for action in meeting_result.action_items:
    print(f"  Action: {action.task} → {action.assignee} by {action.deadline} [{action.priority}]")


> **What just happened?** We built a ValidatedOutput class that: (1) sends a prompt with the Pydantic schema, (2) parses the response, (3) validates with Pydantic, (4) if validation fails, tells the AI what went wrong and retries . The MeetingNotes test shows nested validation — action items inside meeting notes, each with regex-validated date format and priority. If the AI returns "Jan 18" instead of "2026-01-18", Pydantic catches it, and the retry prompt says "deadline must match YYYY-MM-DD format." This is the production pattern: never trust, always validate, auto-retry.


### Step 4 · Gemini JSON Mode — Guaranteed Valid JSON

Instead of asking nicely for JSON, FORCE the model to only output JSON.

**`part4_json_mode.py`** · _Block 5 of 12_

GEMINI JSON MODE: Model is FORCED to output JSON — No markdown fences. No extra text. Just JSON.


In [ ]:
# =============================================
# GEMINI JSON MODE: Model is FORCED to output JSON
# No markdown fences. No extra text. Just JSON.
# =============================================

import google.generativeai as genai
from google.generativeai import protos

# Define the schema as a Gemini-native schema object
schema = protos.Schema(
    type=protos.Type.OBJECT,
    properties={
        "product": protos.Schema(type=protos.Type.STRING),
        "brand": protos.Schema(type=protos.Type.STRING),
        "price": protos.Schema(type=protos.Type.NUMBER),
        "rating": protos.Schema(type=protos.Type.INTEGER),
        "sentiment": protos.Schema(
            type=protos.Type.STRING,
            enum=["positive", "negative", "mixed"]
        ),
        "pros": protos.Schema(
            type=protos.Type.ARRAY,
            items=protos.Schema(type=protos.Type.STRING)
        ),
        "cons": protos.Schema(
            type=protos.Type.ARRAY,
            items=protos.Schema(type=protos.Type.STRING)
        ),
    },
    required=["product", "brand", "rating", "sentiment", "pros", "cons"],
)

# Create model with JSON mode enabled
model = genai.GenerativeModel(
    "gemini-2.0-flash",
    generation_config={
        "response_mime_type": "application/json",  # ← THE MAGIC LINE
        "response_schema": schema,                 # ← enforce this shape
        "temperature": 0.1,
    },
)

# No need to say "return JSON" in the prompt!
review = "The Fire-Boltt Phoenix Ultra smartwatch is decent for ₹1,799. Good display, tracks steps well. But GPS is inaccurate and battery only lasts 3 days. 3/5."

result = model.generate_content(f"Analyze this review: {review}")
data = json.loads(result.text)

print("Gemini JSON Mode output:")
print(json.dumps(data, indent=2))
print(f"\n✅ Guaranteed valid JSON! No cleaning needed.")
print(f"   Product: {data['product']}")
print(f"   Sentiment: {data['sentiment']} (from enum — always valid)")


> **What just happened?** Two magic config lines: response_mime_type: "application/json" and response_schema . The model is now physically constrained to only output JSON matching our schema. No markdown fences, no extra text, no "Here is the JSON:" prefix. The enum on sentiment means it can ONLY return "positive", "negative", or "mixed" — never "somewhat positive" or "mixed-negative." This is 100% reliable JSON output.


### Step 5 · Function Calling — AI Decides Which Tool to Use

The most powerful pattern: define functions, and the AI decides when to call them.

**`part5_function_calling.py`** · _Block 6 of 12_

FUNCTION CALLING: AI decides which tool to use — Define tools → AI picks the right one → your


In [ ]:
# =============================================
# FUNCTION CALLING: AI decides which tool to use
# Define tools → AI picks the right one → your
# code executes it → AI uses the result.
# =============================================

import google.generativeai as genai

# ── Step 1: Define your tools as Python functions ──

def get_product_price(product_name: str, store: str = "amazon") -> dict:
    """Look up the current price of a product."""
    # In production, this would call a real API
    prices = {
        "iphone 16": {"amazon": 79900, "flipkart": 78999},
        "oneplus 13": {"amazon": 57999, "flipkart": 56999},
        "samsung s24": {"amazon": 74999, "flipkart": 73499},
    }
    product = prices.get(product_name.lower(), {})
    price = product.get(store.lower(), None)
    return {"product": product_name, "store": store, "price_inr": price}

def compare_products(product_a: str, product_b: str) -> dict:
    """Compare two products side by side."""
    return {
        "comparison": f"Comparing {product_a} vs {product_b}",
        "product_a": get_product_price(product_a),
        "product_b": get_product_price(product_b),
    }

def check_availability(product_name: str, pincode: str) -> dict:
    """Check if a product is available for delivery to a pincode."""
    return {"product": product_name, "pincode": pincode, "available": True, "delivery_days": 3}

# ── Step 2: Create the model with tools ──
model = genai.GenerativeModel(
    "gemini-2.0-flash",
    tools=[get_product_price, compare_products, check_availability],
)

# ── Step 3: Chat — the AI decides which tool to use! ──
chat = model.start_chat()

test_messages = [
    "What's the price of iPhone 16 on Amazon?",
    "Compare OnePlus 13 and Samsung S24 for me.",
    "Can I get an iPhone 16 delivered to pincode 500081?",
    "Hi, how are you?",  # No tool needed — AI should just chat!
]

for msg in test_messages:
    print(f"\nUser: {msg}")
    response = chat.send_message(msg)
    
    # Check if the AI called a function
    for part in response.parts:
        if hasattr(part, "function_call") and part.function_call:
            fc = part.function_call
            print(f"  AI called: {fc.name}({dict(fc.args)})")
            
            # Execute the function
            func = {"get_product_price": get_product_price,
                    "compare_products": compare_products,
                    "check_availability": check_availability}[fc.name]
            result = func(**dict(fc.args))
            print(f"  Tool result: {result}")
            
            # Send result back to AI for final response
            response = chat.send_message(
                genai.protos.Content(parts=[genai.protos.Part(
                    function_response=genai.protos.FunctionResponse(
                        name=fc.name, response={"result": result}
                    )
                )])
            )
    
    print(f"  AI: {response.text}")


> **What just happened?** We defined 3 Python functions (get_product_price, compare_products, check_availability) and gave them to Gemini as "tools." For each user message, the AI automatically decided: (1) "What's the price of iPhone 16?" → call get_product_price("iphone 16", "amazon") , (2) "Compare OnePlus 13 and Samsung S24" → call compare_products("OnePlus 13", "Samsung S24") , (3) "Hi, how are you?" → no tool needed, just chat. The AI reads your Python function signatures and docstrings to understand what each tool does. This is the foundation of AI agents (Module 9) and MCP (Module 8).


### Step 6 · Project: AI-Powered Product API

Combine JSON mode + Pydantic + function calling into one production endpoint.

**`project_product_api.py`** · _Block 7 of 12_

FULL PROJECT: AI-Powered Product Info API — Natural language in → validated JSON out


In [ ]:
# =============================================
# FULL PROJECT: AI-Powered Product Info API
# Natural language in → validated JSON out
# Uses: Pydantic + JSON mode + function calling
# =============================================

from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum

# ── Schemas ──
class IntentType(str, Enum):
    price_check = "price_check"
    comparison = "comparison"
    availability = "availability"
    review_summary = "review_summary"
    general_question = "general_question"

class UserIntent(BaseModel):
    intent: IntentType
    products: list[str] = Field(description="Product names mentioned")
    store: Optional[str] = Field(default=None, description="Store name if mentioned")
    pincode: Optional[str] = Field(default=None, pattern="^\d{6}$")

class APIResponse(BaseModel):
    success: bool
    intent: str
    data: dict
    message: str = Field(max_length=200)

# ── The pipeline ──
class ProductAPI:
    """Natural language → structured API response."""
    
    def __init__(self):
        self.validator = ValidatedOutput(max_retries=2)
        self.chat_model = genai.GenerativeModel(
            "gemini-2.0-flash",
            tools=[get_product_price, compare_products, check_availability],
        )
    
    def classify_intent(self, query: str) -> UserIntent:
        """Step 1: What does the user want?"""
        return self.validator.generate(
            f'Classify this user query: "{query}"', UserIntent
        )
    
    def execute(self, query: str) -> APIResponse:
        """Full pipeline: classify → route → execute → validate."""
        
        # Step 1: Classify intent
        print(f"  Classifying intent...")
        intent = self.classify_intent(query)
        print(f"  Intent: {intent.intent.value}, Products: {intent.products}")
        
        # Step 2: Route to the right function
        if intent.intent == IntentType.price_check:
            data = get_product_price(intent.products[0], intent.store or "amazon")
            msg = f"Price for {intent.products[0]}: ₹{data['price_inr']:,}" if data["price_inr"] else "Product not found"
        
        elif intent.intent == IntentType.comparison:
            data = compare_products(intent.products[0], intent.products[1] if len(intent.products) > 1 else "unknown")
            msg = f"Comparison between {intent.products[0]} and {intent.products[1] if len(intent.products) > 1 else '?'}"
        
        elif intent.intent == IntentType.availability:
            data = check_availability(intent.products[0], intent.pincode or "500001")
            msg = f"{'Available' if data['available'] else 'Not available'} — delivery in {data['delivery_days']} days"
        
        else:
            data = {"query": query}
            msg = "General question — routed to chat model"
        
        return APIResponse(success=True, intent=intent.intent.value, data=data, message=msg)

# ── Run it! ──
api = ProductAPI()

queries = [
    "How much is the iPhone 16 on Flipkart?",
    "Compare OnePlus 13 vs Samsung S24",
    "Can I get a Samsung S24 delivered to 500081?",
    "What's the best phone under 50K?",
]

for q in queries:
    print(f"\nQuery: \"{q}\"")
    result = api.execute(q)
    print(f"  Response: {result.message}")
    print(f"  JSON: {result.model_dump_json(indent=2)[:150]}...")


> **What just happened?** We built a complete AI-powered API: natural language queries go in → classified into intents (price_check, comparison, availability) using Pydantic validation → routed to the right function → result formatted as a validated APIResponse object. Every step is validated. Every output is typed. No free text ever reaches the app. This is the pattern used by production AI assistants: classify → route → execute → validate → respond.


### Step 7 · The 4-Tier Reliability Hierarchy — Choose the Right Level

2025 changed everything. All three major providers now guarantee schema compliance via constrained decoding.

**`constrained_decoding.py`** · _Block 8 of 12_

HOW CONSTRAINED DECODING WORKS — Your JSON Schema → converted to Context-Free Grammar (CFG)


In [ ]:
# =============================================
# HOW CONSTRAINED DECODING WORKS
# =============================================
# Your JSON Schema → converted to Context-Free Grammar (CFG)
# At EVERY token generation step:
#   1. Model proposes next token probabilities
#   2. Grammar mask sets INVALID tokens to probability 0
#   3. Model can ONLY pick valid tokens
# Result: 100% schema compliance (mathematically impossible to violate)

# OpenAI: 100% on their benchmark (vs <40% without strict mode)
# Claude: Uses grammar-constrained sampling with 24-hour cache
# Gemini: responseSchema with response_mime_type enforcement

# TRADE-OFF: ~10-15% reasoning degradation on complex tasks
# (Tam et al. "Let Me Speak Freely?" found structured output
#  can change the SUBSTANCE of answers, not just format)
# Solution: Use structured CoT (Lesson 3.3) — reason freely,
#           then extract structured data from the reasoning.


> **What just happened?** Constrained decoding converts your schema into a grammar that masks invalid tokens during generation. OpenAI achieved 100% compliance on their eval benchmark. Claude and Gemini use the same technique. The trade-off: forcing structure can degrade reasoning by 10-15% on complex tasks. The fix: separate reasoning from extraction (structured CoT from Lesson 3.3).


### Step 8 · Claude & OpenAI Structured Outputs — Native APIs

The two APIs Indian production teams use most. Both now support constrained decoding.

**`openai_structured.py`** · _Block 9 of 12_

OPENAI STRUCTURED OUTPUTS (strict: true)


In [ ]:
# =============================================
# OPENAI STRUCTURED OUTPUTS (strict: true)
# =============================================
from pydantic import BaseModel
from openai import OpenAI

class ProductReview(BaseModel):
    product_name: str
    sentiment: str   # "positive", "negative", "neutral"
    rating: int       # 1-5
    key_points: list[str]

client = OpenAI()
completion = client.chat.completions.parse(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Review: The OnePlus 13 camera is amazing but battery drains fast"}],
    response_format=ProductReview,  # Pydantic model → auto strict mode
)

# CRITICAL: Always check for refusals and truncation
if completion.choices[0].message.refusal:
    print("Model refused:", completion.choices[0].message.refusal)
elif completion.choices[0].finish_reason == "length":
    print("Output truncated — increase max_tokens")
else:
    result = completion.choices[0].message.parsed
    print(result.product_name)  # "OnePlus 13"
    print(result.sentiment)     # "mixed"


**`claude_structured.py`** · _Block 10 of 12_

CLAUDE STRUCTURED OUTPUTS (output_config)


In [ ]:
# =============================================
# CLAUDE STRUCTURED OUTPUTS (output_config)
# =============================================
import anthropic

client = anthropic.Anthropic()

# Method 1: output_config with json_schema (GA 2026)
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string"},
                    "sentiment": {"type": "string"},
                    "rating": {"type": "integer"}
                },
                "required": ["product_name", "sentiment", "rating"],
                "additionalProperties": False
            }
        }
    },
    messages=[{"role": "user", "content": "Review: Tata Nexon EV is great value but charging infra is poor"}]
)
data = json.loads(response.content[0].text)

# Method 2: tool_choice forced (works with extended thinking)
# When you need reasoning + structured output, use tool-as-schema:
response2 = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    tools=[{
        "name": "extract_review",
        "description": "Extract structured review data",
        "input_schema": {... same schema ...}
    }],
    tool_choice={"type": "tool", "name": "extract_review"},
    messages=[{"role": "user", "content": "..."}]
)


> **What just happened?** All three providers now offer constrained decoding — mathematically guaranteed schema compliance. OpenAI uses .parse() with Pydantic directly. Claude uses output_config for simple extraction or tool_choice forced mode when you need reasoning. Critical: Always check for refusals (OpenAI) and truncation (all providers).


### Step 9 · The instructor Library — Production-Grade Extraction

6M+ monthly downloads. Pydantic + retry for any LLM. OpenAI cited it as inspiration.

**`instructor_demo.py`** · _Block 11 of 12_

pip install instructor


In [ ]:
# pip install instructor
import instructor
from pydantic import BaseModel, field_validator

class JobPosting(BaseModel):
    title: str
    company: str
    location: str
    salary_min: int
    salary_max: int
    remote: bool

    @field_validator('salary_max')
    @classmethod
    def max_gt_min(cls, v, info):
        if 'salary_min' in info.data and v < info.data['salary_min']:
            raise ValueError('salary_max must be >= salary_min')
        return v

# Works with ANY provider — swap one line to switch
client = instructor.from_provider("openai/gpt-4o-mini")
# client = instructor.from_provider("anthropic/claude-sonnet-4-6")
# client = instructor.from_provider("google/gemini-2.5-flash")

job = client.chat.completions.create(
    response_model=JobPosting,
    messages=[{"role": "user",
        "content": "Hiring: Senior Python Dev at Flipkart Bangalore, 25-40 LPA, hybrid"}],
    max_retries=3,  # On validation error → sends error back to LLM → retries
)
print(job.model_dump_json(indent=2))
# {"title":"Senior Python Developer","company":"Flipkart",
#  "location":"Bangalore","salary_min":2500000,"salary_max":4000000,
#  "remote":false}


> **What just happened?** instructor patches any LLM SDK to add a response_model parameter. Define a Pydantic model → call the API → get a validated Python object. On validation failure, it feeds the error back to the LLM and retries automatically. Supports 15+ providers with one-line switching. This is the production standard for structured extraction.


### Step 10 · India-Specific Extraction — GSTIN, PAN, Aadhaar & Hinglish

Indian document formats need regex + algorithmic validation. Regex alone is not enough.

**`indian_extraction.py`** · _Block 12 of 12_


In [ ]:
import re
from pydantic import BaseModel, field_validator

class IndianBusinessEntity(BaseModel):
    """Extract Indian business identifiers from invoices/documents."""
    company_name: str
    gstin: str | None = None
    pan: str | None = None
    state_code: str | None = None

    @field_validator('gstin')
    @classmethod
    def validate_gstin(cls, v):
        """GSTIN: 15 chars — 2-digit state + PAN + entity + Z + check"""
        if v and not re.match(
            r'^([0][1-9]|[1-2][0-9]|[3][0-7])[A-Z]{5}[0-9]{4}[A-Z][1-9A-Z]Z[0-9A-Z]$', v
        ):
            raise ValueError('Invalid GSTIN format. Must be 15 chars with state code 01-37')
        return v

    @field_validator('pan')
    @classmethod
    def validate_pan(cls, v):
        """PAN: AAAAA9999A — 4th char = holder type (P/C/H/F/T...)"""
        if v and not re.match(r'^[A-Z]{3}[ABCFGHLJPTF][A-Z][0-9]{4}[A-Z]$', v):
            raise ValueError('Invalid PAN. 4th char must be holder type (P/C/H/F/T)')
        return v

    @field_validator('gstin')
    @classmethod
    def cross_validate_gstin_pan(cls, v, info):
        """GSTIN positions 3-12 must match PAN"""
        if v and 'pan' in info.data and info.data['pan']:
            if v[2:12] != info.data['pan']:
                raise ValueError(f'GSTIN contains PAN {v[2:12]} but PAN field is {info.data["pan"]}')
        return v

# Usage with instructor (any provider)
entity = client.chat.completions.create(
    response_model=IndianBusinessEntity,
    messages=[{"role": "user",
        "content": "Invoice from: Tata Consultancy Services, GSTIN: 27AAACT2727Q1ZV, PAN: AAACT2727Q"}],
    max_retries=2,
)


> **What just happened?** Indian business identifiers need regex + algorithmic validation . GSTIN embeds the PAN at positions 3-12, enabling cross-validation. PAN's 4th character indicates holder type. Aadhaar requires Verhoeff checksum. Always combine LLM extraction with Pydantic validators — the LLM finds the data, Pydantic catches the errors.


---

## Wrap-up

You've walked through all 12 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
